In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

FEAT = Path('../features')
MODELS = Path('../models')

# Load data and the fused model
print("Loading data...")
rad = pd.read_csv(FEAT / 'radiomics.csv')
deep = pd.read_csv(FEAT / 'deep.csv')
merged = rad.merge(deep, on=['image_id', 'split', 'class'])

bundle = joblib.load(MODELS / 'fused.pkl')
scaler = bundle['scaler']
selector = bundle['selector']
all_cols = bundle['all_cols']

# Build the training-set feature matrix in selected-feature space
print("Building similarity index from training set...")
train = merged[merged['split'] == 'Training'].reset_index(drop=True)
X_train_full = train[all_cols].values
X_train_scaled = scaler.transform(X_train_full)
X_train_selected = selector.transform(X_train_scaled)

# Save the feature matrix + image paths + classes
similarity_index = {
    'features': X_train_selected,                  # (5600, 300)
    'image_ids': train['image_id'].tolist(),
    'classes': train['class'].tolist(),
    'splits': train['split'].tolist(),
}

joblib.dump(similarity_index, MODELS / 'similarity_index.pkl')

print(f"Saved similarity index:")
print(f"  Feature matrix shape: {X_train_selected.shape}")
print(f"  Number of training images indexed: {len(train)}")
print(f"  File size: {(MODELS / 'similarity_index.pkl').stat().st_size / 1024 / 1024:.1f} MB")

Loading data...
Building similarity index from training set...
Saved similarity index:
  Feature matrix shape: (5600, 300)
  Number of training images indexed: 5600
  File size: 13.0 MB
